<a href="https://colab.research.google.com/github/EUNTELLA/baseball/blob/main/0810result.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [4]:
import shutil
from pathlib import Path

repo_path = "/content/baseball"
# Corrected URL for downloading the main branch as a zip file
repo_zip_url = "https://github.com/EUNTELLA/baseball/archive/refs/heads/main.zip"

# Remove the directory if it already exists to ensure a clean setup
if Path(repo_path).exists():
    shutil.rmtree(repo_path)
    print(f"Removed existing directory: {repo_path}")

# Create the directory
Path(repo_path).mkdir(parents=True, exist_ok=True)

# Download the repository as a zip file
zip_filename = Path(repo_path) / "baseball_main.zip"
!wget -O "{zip_filename}" "{repo_zip_url}"

# Unzip the contents
# Ensure the zip file exists before attempting to unzip
if zip_filename.exists():
    !unzip -o "{zip_filename}" -d "{repo_path}"

    # The content will be extracted into a subdirectory named 'baseball-main'
    # Move contents up to the 'baseball' directory
    extracted_path = Path(repo_path) / "baseball-main"
    if extracted_path.exists():
        for item in extracted_path.iterdir():
            shutil.move(str(item), repo_path)
        shutil.rmtree(extracted_path)
    else:
        print(f"Warning: Extracted directory '{extracted_path}' not found.")
else:
    print(f"Error: Zip file not downloaded from {repo_zip_url}.")

# Change into the repository directory
# Only change directory if the repo_path exists and seems to contain the repo
if Path(repo_path).exists() and any(Path(repo_path).iterdir()):
    %cd "{repo_path}"
else:
    print(f"Error: Repository directory '{repo_path}' is empty or does not exist.")

--2026-08-11 09:24:54--  https://github.com/EUNTELLA/baseball/archive/refs/heads/main.zip
Resolving github.com (github.com)... 140.82.114.4
Connecting to github.com (github.com)|140.82.114.4|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://codeload.github.com/EUNTELLA/baseball/zip/refs/heads/main [following]
--2026-08-11 09:24:55--  https://codeload.github.com/EUNTELLA/baseball/zip/refs/heads/main
Resolving codeload.github.com (codeload.github.com)... 140.82.112.9
Connecting to codeload.github.com (codeload.github.com)|140.82.112.9|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: unspecified [application/zip]
Saving to: ‘/content/baseball/baseball_main.zip’

/content/baseball/b     [   <=>              ]   3.79M  9.13MB/s    in 0.4s    

2026-08-11 09:24:56 (9.13 MB/s) - ‘/content/baseball/baseball_main.zip’ saved [3980070]

Archive:  /content/baseball/baseball_main.zip
1280458b326f01de34ca03b4fd0e33486e10ca50
   creating: 

In [5]:
!mkdir -p /content/baseball/open
!unzip -o "/content/drive/MyDrive/open.zip" -d /content/baseball/open

Archive:  /content/drive/MyDrive/open.zip
  inflating: /content/baseball/open/baseline_submit.zip  
  inflating: /content/baseball/open/data/sample_submission.csv  
  inflating: /content/baseball/open/data/test.csv  
  inflating: /content/baseball/open/data/trackman_history.csv  
  inflating: /content/baseball/open/data/train.csv  
  inflating: /content/baseball/open/data_description.md  


In [6]:
import sys
from pathlib import Path

# The common.py module is now located inside /content/baseball/0826
experiment_dir = Path("/content/baseball/0826")

# Add the experiment directory to sys.path
if str(experiment_dir) not in sys.path:
    sys.path.insert(0, str(experiment_dir))

print((experiment_dir / "common.py").exists())
print(Path("/content/baseball/open/data/train.csv").exists())

True
True


# 05. RandomForest 전체 재학습 및 제출 ZIP 생성

#2019~2024년 전체 학습 데이터로 운영진 RandomForest를 재학습합니다. 동일 모델로 원본 확률 제출본과 검증에서 선택한 `-0.01056616` 보정 제출본을 각각 생성하고, 샘플 데이터로 패키지 구조와 추론을 검증합니다.
## 1. 평가 서버와 동일한 주요 패키지 설치

#이 셀은 모델 직렬화 호환성을 위해 운영진 베이스라인 제출본과 동일한 버전을 설치합니다. 설치 후 런타임 재시작 안내가 나오면 재시작한 뒤 이 셀 다음부터 실행하세요.

In [1]:
%pip install -q scikit-learn==1.8.0 joblib==1.5.3 pandas==2.3.3

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 64.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 82.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.


## 1. 경로와 설정 확인

In [7]:
from __future__ import annotations

import json
import shutil
import subprocess
import sys
import time
import zipfile
from pathlib import Path

import joblib
import pandas as pd
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder

ROOT = Path('/content/baseball') if Path('/content/baseball').exists() else Path.cwd()
DATA_DIR = ROOT / 'open' / 'data'
EXPERIMENT_DIR = ROOT / '0826'
RESULTS_DIR = EXPERIMENT_DIR / 'results'
BUILD_DIR = RESULTS_DIR / 'final_submission_build'

TRAIN_PATH = DATA_DIR / 'train.csv'
TEST_PATH = DATA_DIR / 'test.csv'
SAMPLE_PATH = DATA_DIR / 'sample_submission.csv'
CALIBRATION_OFFSET = -0.01056616

for path in [TRAIN_PATH, TEST_PATH, SAMPLE_PATH]:
    if not path.exists():
        raise FileNotFoundError(f'필수 파일이 없습니다: {path}')

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print('root:', ROOT)
print('sklearn:', sklearn.__version__)
print('pandas:', pd.__version__)
print('calibration offset:', CALIBRATION_OFFSET)

root: /content/baseball
sklearn: 1.8.0
pandas: 2.3.3
calibration offset: -0.01056616


## 2. 전체 학습 데이터로 RandomForest 재학습

Colab CPU에서 수 분 이상 걸릴 수 있습니다. 생성되는 모델은 검증 Fold 모델이 아니라 2019~2024년 전체 데이터로 학습한 최종 모델입니다.

In [8]:
ID_COL = 'row_id'
TARGET_COL = 'control_success'
CAT_COLS = ['top_bottom', 'game_type', 'base_state']

header = pd.read_csv(TEST_PATH, encoding='utf-8-sig', nrows=0)
features = [c for c in header.columns if c != ID_COL]
numeric_cols = [c for c in features if c not in CAT_COLS]

train = pd.read_csv(
    TRAIN_PATH,
    encoding='utf-8-sig',
    usecols=features + [TARGET_COL],
)
print(f'train={train.shape}, features={len(features)}')
print(f'seasons={train["season"].min()}~{train["season"].max()}')
print(f'target rate={train[TARGET_COL].mean():.8f}')

preprocessor = ColumnTransformer([
    (
        'cat',
        OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1),
        CAT_COLS,
    ),
    ('num', SimpleImputer(strategy='median'), numeric_cols),
])

model = Pipeline([
    ('pre', preprocessor),
    (
        'clf',
        RandomForestClassifier(
            n_estimators=100,
            max_depth=10,
            min_samples_leaf=200,
            n_jobs=-1,
            random_state=42,
        ),
    ),
])

started_at = time.perf_counter()
model.fit(train[features], train[TARGET_COL])
train_seconds = time.perf_counter() - started_at
print(f'전체 재학습 완료: {train_seconds:.1f}초')

MODEL_PATH = RESULTS_DIR / 'rf_full_2019_2024.pkl'
joblib.dump(model, MODEL_PATH, compress=3)
print(f'model saved: {MODEL_PATH} ({MODEL_PATH.stat().st_size / 1024**2:.2f} MiB)')
del train

train=(1475092, 48), features=47
seasons=2019~2024
target rate=0.52376598
전체 재학습 완료: 504.5초
model saved: /content/baseball/0826/results/rf_full_2019_2024.pkl (3.77 MiB)


## 3. 원본 및 보정 제출 ZIP 생성


In [9]:
REQUIREMENTS = (
    f'scikit-learn=={sklearn.__version__}\n'
    f'joblib=={joblib.__version__}\n'
    f'pandas=={pd.__version__}\n'
)

def make_inference_script(offset: float) -> str:
    return f'''import os

import joblib
import pandas as pd

ID_COL = "row_id"
TARGET_COL = "control_success"
CALIBRATION_OFFSET = {offset!r}


def main():
    test_path = "./data/test.csv"
    sample_path = "./data/sample_submission.csv"
    model_path = "./model/rf.pkl"
    output_path = "./output/submission.csv"

    model = joblib.load(model_path)
    test = pd.read_csv(test_path, encoding="utf-8-sig")
    submission = pd.read_csv(sample_path, encoding="utf-8-sig")

    if ID_COL not in test.columns:
        raise ValueError("test.csv에 row_id가 없습니다.")
    if list(submission.columns[:2]) != [ID_COL, TARGET_COL]:
        raise ValueError("sample_submission.csv 컬럼이 올바르지 않습니다.")
    if test[ID_COL].duplicated().any():
        raise ValueError("test.csv에 중복 row_id가 있습니다.")

    features = test.drop(columns=[ID_COL])
    predictions = model.predict_proba(features)[:, 1]
    predictions = pd.Series(predictions + CALIBRATION_OFFSET).clip(0.0, 1.0)

    prediction_by_id = pd.Series(
        predictions.to_numpy(),
        index=test[ID_COL].to_numpy(),
    )
    aligned = prediction_by_id.reindex(submission[ID_COL])
    if aligned.isna().any():
        raise ValueError("예측값이 없는 row_id가 있습니다.")

    submission[TARGET_COL] = aligned.to_numpy()
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    submission.to_csv(output_path, index=False, encoding="utf-8")
    print("Saved:", output_path, "rows=", len(submission))
    print("Prediction mean:", float(submission[TARGET_COL].mean()))
    print("Calibration offset:", CALIBRATION_OFFSET)


if __name__ == "__main__":
    main()
'''

def build_submission_zip(name: str, offset: float) -> Path:
    package_dir = BUILD_DIR / name
    if package_dir.exists():
        shutil.rmtree(package_dir)
    (package_dir / 'model').mkdir(parents=True)

    shutil.copy2(MODEL_PATH, package_dir / 'model' / 'rf.pkl')
    (package_dir / 'script.py').write_text(
        make_inference_script(offset), encoding='utf-8'
    )
    (package_dir / 'requirements.txt').write_text(
        REQUIREMENTS, encoding='utf-8'
    )

    zip_path = RESULTS_DIR / f'{name}.zip'
    if zip_path.exists():
        zip_path.unlink()
    with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
        for file_path in sorted(package_dir.rglob('*')):
            if file_path.is_file():
                archive.write(file_path, file_path.relative_to(package_dir))

    return zip_path

raw_zip = build_submission_zip('submit_rf_raw', 0.0)
calibrated_zip = build_submission_zip(
    'submit_rf_calibrated', CALIBRATION_OFFSET
)
print(raw_zip, f'{raw_zip.stat().st_size / 1024**2:.2f} MiB')
print(calibrated_zip, f'{calibrated_zip.stat().st_size / 1024**2:.2f} MiB')

/content/baseball/0826/results/submit_rf_raw.zip 3.75 MiB
/content/baseball/0826/results/submit_rf_calibrated.zip 3.75 MiB


## 4. ZIP 구조 및 샘플 추론 검증

각 ZIP을 별도 폴더에 해제한 뒤 평가 서버처럼 `data/`를 추가하고 실제 `script.py`를 실행합니다.

In [10]:
def validate_submission(zip_path: Path) -> dict:
    validation_dir = RESULTS_DIR / 'validation' / zip_path.stem
    if validation_dir.exists():
        shutil.rmtree(validation_dir)
    validation_dir.mkdir(parents=True)

    with zipfile.ZipFile(zip_path) as archive:
        names = sorted(archive.namelist())
        archive.extractall(validation_dir)

    expected = ['model/rf.pkl', 'requirements.txt', 'script.py']
    if names != expected:
        raise ValueError(f'ZIP 구조 불일치: {names}')

    data_dir = validation_dir / 'data'
    data_dir.mkdir()
    shutil.copy2(TEST_PATH, data_dir / 'test.csv')
    shutil.copy2(SAMPLE_PATH, data_dir / 'sample_submission.csv')

    completed = subprocess.run(
        [sys.executable, 'script.py'],
        cwd=validation_dir,
        capture_output=True,
        text=True,
        check=True,
    )
    output_path = validation_dir / 'output' / 'submission.csv'
    output = pd.read_csv(output_path)
    if len(output) != len(pd.read_csv(TEST_PATH)):
        raise ValueError('출력 행 수가 test.csv와 다릅니다.')
    if not output[TARGET_COL].between(0.0, 1.0).all():
        raise ValueError('예측 확률 범위가 0~1을 벗어났습니다.')

    result = {
        'zip': str(zip_path),
        'members': names,
        'rows': len(output),
        'prediction_mean': float(output[TARGET_COL].mean()),
        'stdout': completed.stdout.strip(),
    }
    print(json.dumps(result, ensure_ascii=False, indent=2))
    return result

validation_results = [
    validate_submission(raw_zip),
    validate_submission(calibrated_zip),
]
(RESULTS_DIR / '05_submission_validation.json').write_text(
    json.dumps(validation_results, ensure_ascii=False, indent=2),
    encoding='utf-8',
)

{
  "zip": "/content/baseball/0826/results/submit_rf_raw.zip",
  "members": [
    "model/rf.pkl",
    "requirements.txt",
    "script.py"
  ],
  "rows": 5,
  "prediction_mean": 0.478338478518757,
  "stdout": "Saved: ./output/submission.csv rows= 5\nPrediction mean: 0.4783384785187571\nCalibration offset: 0.0"
}
{
  "zip": "/content/baseball/0826/results/submit_rf_calibrated.zip",
  "members": [
    "model/rf.pkl",
    "requirements.txt",
    "script.py"
  ],
  "rows": 5,
  "prediction_mean": 0.46777231851875706,
  "stdout": "Saved: ./output/submission.csv rows= 5\nPrediction mean: 0.46777231851875706\nCalibration offset: -0.01056616"
}


692

## 5. 결과를 Google Drive에 보관

Drive가 마운트되어 있으면 두 제출 ZIP과 검증 기록을 복사합니다. 대회에는 우선 `submit_rf_calibrated.zip`을 신규 실험 제출본으로 사용하고, 기존 Public 549.5119 제출 결과는 안전 기준점으로 유지합니다.

In [12]:
DRIVE_OUTPUT_DIR = Path('/content/drive/MyDrive/baseball-results/final')
if Path('/content/drive/MyDrive').exists():
    DRIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    for path in [
        raw_zip,
        calibrated_zip,
        RESULTS_DIR / '05_submission_validation.json',
    ]:
        shutil.copy2(path, DRIVE_OUTPUT_DIR / path.name)
    print('Drive 저장 완료:', DRIVE_OUTPUT_DIR)
else:
    print('Drive가 마운트되어 있지 않습니다.')
    print('로컬 결과 경로:', RESULTS_DIR)

Drive 저장 완료: /content/drive/MyDrive/baseball-results/final
